In [1]:
from pathlib import Path
import sys

# Permite importar src quando o notebook é executado dentro de notebooks/
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import sionna
import torch

from src.config import load_config
from src.channel import generate_channel, compute_rates
from src.simulation import jains_fairness_index
from src.schedulers import ProportionalFair

cfg = load_config()

print("Python:", sys.version)
print("Executável:", sys.executable)
print("Sionna:", sionna.__version__)
print("PyTorch:", torch.__version__)
print("Beta PF:", cfg.pf_beta)
print("TTIs:", cfg.num_ttis)
print("Cargas:", cfg.user_counts)

assert sys.executable.startswith("/opt/venv/")
assert sionna.__version__ == "2.0.1"
assert cfg.pf_beta == 0.98

print("\nAMBIENTE DOCKER VALIDADO.")

Python: 3.11.16 (main, Aug 13 2026, 19:40:50) [GCC 14.2.0]
Executável: /opt/venv/bin/python3.11
Sionna: 2.0.1
PyTorch: 2.13.0+cu130
Beta PF: 0.98
TTIs: 10000
Cargas: [2, 4, 8, 16, 32]

AMBIENTE DOCKER VALIDADO.


In [2]:
# ============================================================
# TESTE 1 — VALIDAÇÃO MATEMÁTICA ISOLADA DO PROPORTIONAL FAIR
# ============================================================

beta = 0.98
pf = ProportionalFair(num_ues=3, beta=beta)

# ------------------------------------------------------------
# Estado inicial
# ------------------------------------------------------------
esperado_inicial = np.array([1.0, 1.0, 1.0])

assert np.allclose(pf.t_hist, esperado_inicial)

print("Histórico inicial:")
print(pf.t_hist)


# ------------------------------------------------------------
# TTI 0
#
# Como todos começam com T=1:
#
# métricas = R/T
#          = [10/1, 20/1, 5/1]
#          = [10, 20, 5]
#
# Portanto o UE 1 deve vencer.
# ------------------------------------------------------------

rates_t0 = np.array([10.0, 20.0, 5.0])

vencedor_t0 = pf.select(tti=0, rates=rates_t0)

print("\nTaxas TTI 0:", rates_t0)
print("Vencedor TTI 0:", vencedor_t0)

assert vencedor_t0 == 1


# ------------------------------------------------------------
# Atualização esperada:
#
# R_t = [0, 20, 0]
#
# T_t = 0.98 * [1,1,1] + 0.02 * [0,20,0]
#
#     = [0.98, 1.38, 0.98]
# ------------------------------------------------------------

pf.update(
    tti=0,
    ue=vencedor_t0,
    rate=rates_t0[vencedor_t0]
)

esperado_apos_t0 = np.array([
    0.98,
    1.38,
    0.98
])

print("\nHistórico calculado:")
print(pf.t_hist)

print("Histórico esperado:")
print(esperado_apos_t0)

assert np.allclose(
    pf.t_hist,
    esperado_apos_t0
)


# ------------------------------------------------------------
# TTI 1
#
# Agora usamos outro vetor de taxas.
#
# O vencedor deve ser exatamente o argmax(R/T).
# ------------------------------------------------------------

rates_t1 = np.array([10.0, 8.0, 5.0])

metricas_esperadas = rates_t1 / esperado_apos_t0
vencedor_esperado = int(np.argmax(metricas_esperadas))

vencedor_real = pf.select(
    tti=1,
    rates=rates_t1
)

print("\nTaxas TTI 1:", rates_t1)
print("Métricas R/T:", metricas_esperadas)
print("Vencedor esperado:", vencedor_esperado)
print("Vencedor da classe:", vencedor_real)

assert vencedor_real == vencedor_esperado

print("\nTESTE MATEMÁTICO DO PF: PASSOU.")

Histórico inicial:
[1. 1. 1.]

Taxas TTI 0: [10. 20.  5.]
Vencedor TTI 0: 1

Histórico calculado:
[0.98 1.38 0.98]
Histórico esperado:
[0.98 1.38 0.98]

Taxas TTI 1: [10.  8.  5.]
Métricas R/T: [10.20408163  5.79710145  5.10204082]
Vencedor esperado: 0
Vencedor da classe: 0

TESTE MATEMÁTICO DO PF: PASSOU.


In [3]:
# ============================================================
# TESTE 2 — PF SOBRE O CANAL REAL DO CARD 1
# ============================================================

def executar_pf(carga: int, seed: int):
    """
    Executa o Proportional Fair usando exatamente o canal
    e o modelo de taxa definidos no Card 1.
    """

    # Gera o canal Rayleigh real do projeto.
    real = generate_channel(
        seed=seed,
        num_ues=carga,
        num_ttis=cfg.num_ttis
    )

    # Converte os coeficientes de canal em taxas instantâneas.
    #
    # Shape:
    # [num_ttis, carga]
    rates = compute_rates(cfg, real.h)

    assert rates.shape == (
        cfg.num_ttis,
        carga
    )

    # Scheduler PF.
    pf = ProportionalFair(
        num_ues=carga,
        beta=cfg.pf_beta
    )

    # Acumuladores.
    bits = np.zeros(carga, dtype=np.float64)
    slots = np.zeros(carga, dtype=np.int64)

    for t in range(cfg.num_ttis):

        # Vetor R_u(t) de todos os UEs no TTI atual.
        rates_t = rates[t]

        # PF escolhe argmax R/T.
        ue = pf.select(
            tti=t,
            rates=rates_t
        )

        assert 0 <= ue < carga

        # UE escalonado recebe todo o recurso do TTI.
        bits[ue] += (
            rates_t[ue]
            * cfg.tti_duration
        )

        slots[ue] += 1

        # Atualiza T_u.
        pf.update(
            tti=t,
            ue=ue,
            rate=rates_t[ue]
        )

    # Throughput médio por UE.
    throughput = (
        bits
        / cfg.sim_duration_per_seed_s
    )

    jfi = jains_fairness_index(
        throughput
    )

    return {
        "carga": carga,
        "seed": seed,
        "throughput": throughput,
        "slots": slots,
        "jfi": jfi,
        "historico_final": pf.t_hist,
    }


resultado = executar_pf(
    carga=4,
    seed=42
)

print("=== PF — CANAL REAL DO CARD 1 ===")

for ue in range(resultado["carga"]):
    print(
        f"UE {ue}: "
        f"{resultado['throughput'][ue] / 1e6:.2f} Mbps | "
        f"{resultado['slots'][ue]} slots"
    )

print(
    "\nThroughput agregado:",
    f"{resultado['throughput'].sum() / 1e6:.2f} Mbps"
)

print(
    "JFI:",
    f"{resultado['jfi']:.6f}"
)

assert np.all(
    np.isfinite(resultado["throughput"])
)

assert np.all(
    resultado["throughput"] >= 0
)

assert 0 < resultado["jfi"] <= 1

assert resultado["slots"].sum() == cfg.num_ttis

print("\nPF INTEGRADO AO CANAL REAL: PASSOU.")

=== PF — CANAL REAL DO CARD 1 ===
UE 0: 19.74 Mbps | 2509 slots
UE 1: 19.59 Mbps | 2505 slots
UE 2: 19.75 Mbps | 2487 slots
UE 3: 19.67 Mbps | 2499 slots

Throughput agregado: 78.75 Mbps
JFI: 0.999990

PF INTEGRADO AO CANAL REAL: PASSOU.


In [4]:
# ============================================================
# TESTE 3 — TODAS AS CARGAS EXIGIDAS PELO CARD 4
# ============================================================

linhas = []

for carga in cfg.user_counts:

    resultado = executar_pf(
        carga=carga,
        seed=42
    )

    linhas.append({
        "carga": carga,
        "seed": 42,
        "jfi": resultado["jfi"],
        "throughput_agregado_Mbps":
            resultado["throughput"].sum() / 1e6,
        "min_slots":
            resultado["slots"].min(),
        "max_slots":
            resultado["slots"].max(),
        "total_slots":
            resultado["slots"].sum(),
    })


df_validacao = pd.DataFrame(linhas)

display(df_validacao)


assert (
    df_validacao["total_slots"]
    == cfg.num_ttis
).all()

assert (
    (df_validacao["jfi"] > 0)
    & (df_validacao["jfi"] <= 1)
).all()


print(
    "\nTODAS AS CARGAS DO CARD 4 "
    "FORAM EXECUTADAS COM SUCESSO."
)

,carga,seed,jfi,throughput_agregado_Mbps,min_slots,max_slots,total_slots
0,2,42,0.999999,68.540460,4998,5002,10000
1,4,42,0.999990,78.748276,2487,2509,10000
2,8,42,0.999986,85.448850,1245,1256,10000
3,16,42,0.999978,88.737094,622,629,10000
4,32,42,0.999946,89.601923,309,316,10000



TODAS AS CARGAS DO CARD 4 FORAM EXECUTADAS COM SUCESSO.


## Conclusão

O scheduler Proportional Fair foi validado em três níveis:

1. A regra de seleção `R(u,t) / T(u,t)` foi verificada diretamente com valores controlados.
2. A atualização do histórico de throughput com `beta = 0.98` foi conferida numericamente.
3. O scheduler foi executado sobre o canal Rayleigh real do Card 1, incluindo todas as cargas previstas no experimento: 2, 4, 8, 16 e 32 UEs.

Todos os testes funcionais passaram sem erros.

A comparação de JFI com Round Robin e Max C/I não foi incluída nesta etapa, pois os schedulers dos Cards 2 e 3 ainda não fazem parte da integração atual. Essa comparação será realizada após a integração dos três schedulers.

In [5]:

# ============================================================
# TESTE 4 — COMPARAÇÃO PROPORTIONAL FAIR vs ROUND ROBIN
# ============================================================

from src.schedulers import RoundRobin

def executar_rr(carga: int, seed: int):
    np.random.seed(seed)
    #tf.random.set_seed(seed)

    cfg_local = cfg

    sim = Simulation(cfg_local)
    scheduler = RoundRobin(num_ues=carga)

    throughput = np.zeros(carga)
    alocacoes = np.zeros(carga, dtype=int)

    for tti in range(cfg_local.tti_per_step):
        rates = sim.estimate_rates()
        ue = scheduler.select(tti, rates)
        rate = float(rates[ue])
        scheduler.update(tti, ue, rate)

        throughput[ue] += rate
        alocacoes[ue] += 1

    fairness = jain_index(throughput)

    return {
        "Throughput total (Mbps)": throughput.sum()/1e6,
        "Jain": fairness,
        "Alocações": alocacoes,
        "Throughput UE": throughput/1e6,
    }

comparacao = []

for carga in cfg.user_counts:
    pf_res = executar_pf(carga, seed=42)
    rr_res = executar_rr(carga, seed=42)

    comparacao.append({
        "Usuários": carga,
        "PF Jain": round(pf_res["Jain"], 3),
        "RR Jain": round(rr_res["Jain"], 3),
        "PF Total": round(pf_res["Throughput total (Mbps)"], 1),
        "RR Total": round(rr_res["Throughput total (Mbps)"], 1),
    })

df_comp = pd.DataFrame(comparacao)
display(df_comp)

print("\nExemplo de distribuição de alocações (10 usuários):")
rr10 = executar_rr(10, seed=42)
print(rr10["Alocações"])


NameError: name 'Simulation' is not defined